# Exp18 — Preflight gate (BLOOM first; Gemma optional)

Decides whether the BLOOM leg of the language grid exists at all. Four checks, in order:

1. **Name tokenization** — how many names per language are single-token at the answer position (native names, and EN names inside native templates).
2. **Agreement-form collisions** — do the two verb forms of a minimal pair share a first token (the failure that killed Arabic on Gemma in Exp15b).
3. **Byte → char/token share conversion** — ROOTS reports *bytes*; this converts to character and token shares relative to English.
4. **IOI baseline accuracy** — forced choice IO vs S at the final position, per language and condition.

Gate (Exp15 rule): a (model, language, condition) cell is usable only if **>= MIN_NAMES single-token names** and **baseline accuracy >= 0.75**.
No patching here. Nothing downstream should be built on a cell that fails.

Caveats: templates and parallel sentences were written by hand and need a native-speaker read before they go in the paper. German is not one of the 46 ROOTS languages, so DE rows on BLOOM are informational only.

In [1]:
# CELL 1 — config + data. Only edit MODELS (paths) and GPU_ID.
import os
GPU_ID = 2
os.environ.setdefault("CUDA_VISIBLE_DEVICES", str(GPU_ID))   # must precede any torch import
import json, random, math
from pathlib import Path
OUT = Path("preflight_results"); OUT.mkdir(exist_ok=True)

MODELS = {
    "bloom-1b7": "bigscience/bloom-1b7",
    "bloom-7b1": "bigscience/bloom-7b1",
    # "gemma-2-9b": "/home/models/gemma-2-9b",
}
def prepend_bos(key): return "gemma" in key      # BLOOM is not trained with a BOS token
LANGS = ["en", "fr", "es", "de", "zh", "ar", "hi"]
SEP = {"zh": ""}                                  # string between context and a name (default " ")
N_PROMPTS, SEED, MIN_NAMES, ACC_GATE = 200, 0, 8, 0.75

# ROOTS byte shares (%) as reported in the ROOTS paper; None = not in my verified list, fill from ROOTS Table 1.
ROOTS_BYTE_SHARE = {"en": 30.03, "zh": 16.16, "fr": 12.9, "es": 10.85, "ar": 4.6,
                    "hi": None, "de": None}       # de: not a ROOTS language

PARALLEL = {
 "en": ["The children went to the market early in the morning to buy fresh fruit.",
        "My sister is reading a long book about the history of science.",
        "If it rains tomorrow, we will stay at home and cook dinner together.",
        "The teacher asked the students to finish their homework before the holiday.",
        "He did not know that the train had already left the station.",
        "Many people think that learning a new language takes a lot of time."],
 "fr": ["Les enfants sont allés au marché tôt le matin pour acheter des fruits frais.",
        "Ma sœur lit un long livre sur l'histoire des sciences.",
        "S'il pleut demain, nous resterons à la maison et nous préparerons le dîner ensemble.",
        "La professeure a demandé aux élèves de finir leurs devoirs avant les vacances.",
        "Il ne savait pas que le train avait déjà quitté la gare.",
        "Beaucoup de gens pensent qu'apprendre une nouvelle langue prend beaucoup de temps."],
 "es": ["Los niños fueron al mercado temprano por la mañana para comprar fruta fresca.",
        "Mi hermana está leyendo un libro largo sobre la historia de la ciencia.",
        "Si mañana llueve, nos quedaremos en casa y cocinaremos la cena juntos.",
        "La profesora pidió a los alumnos que terminaran la tarea antes de las vacaciones.",
        "Él no sabía que el tren ya había salido de la estación.",
        "Mucha gente piensa que aprender un idioma nuevo requiere mucho tiempo."],
 "de": ["Die Kinder gingen früh am Morgen zum Markt, um frisches Obst zu kaufen.",
        "Meine Schwester liest ein langes Buch über die Geschichte der Wissenschaft.",
        "Wenn es morgen regnet, bleiben wir zu Hause und kochen zusammen das Abendessen.",
        "Die Lehrerin bat die Schüler, ihre Hausaufgaben vor den Ferien zu beenden.",
        "Er wusste nicht, dass der Zug den Bahnhof bereits verlassen hatte.",
        "Viele Leute glauben, dass das Lernen einer neuen Sprache viel Zeit braucht."],
 "zh": ["孩子们一大早就去市场买新鲜水果。",
        "我姐姐正在读一本关于科学史的长篇书籍。",
        "如果明天下雨，我们就待在家里一起做晚饭。",
        "老师要求学生在假期之前完成作业。",
        "他不知道火车已经离开了车站。",
        "很多人认为学习一门新语言需要花很多时间。"],
 "ar": ["ذهب الأطفال إلى السوق في الصباح الباكر لشراء الفاكهة الطازجة.",
        "تقرأ أختي كتابا طويلا عن تاريخ العلوم.",
        "إذا أمطرت غدا فسوف نبقى في المنزل ونطبخ العشاء معا.",
        "طلبت المعلمة من الطلاب إنهاء واجباتهم قبل العطلة.",
        "لم يكن يعلم أن القطار قد غادر المحطة بالفعل.",
        "يعتقد كثير من الناس أن تعلم لغة جديدة يستغرق وقتا طويلا."],
 "hi": ["बच्चे सुबह जल्दी ताज़े फल खरीदने बाज़ार गए।",
        "मेरी बहन विज्ञान के इतिहास पर एक लंबी किताब पढ़ रही है।",
        "अगर कल बारिश हुई तो हम घर पर रहेंगे और साथ मिलकर खाना बनाएँगे।",
        "शिक्षक ने छात्रों से छुट्टी से पहले अपना गृहकार्य पूरा करने को कहा।",
        "उसे नहीं पता था कि ट्रेन पहले ही स्टेशन से जा चुकी थी।",
        "बहुत से लोग मानते हैं कि नई भाषा सीखने में बहुत समय लगता है।"],
}

NAMES = {
 "en": ["Mary","John","Tom","Anna","James","Sarah","David","Emily","Michael","Laura","Peter","Julia"],
 "fr": ["Marie","Jean","Pierre","Sophie","Lucas","Emma","Hugo","Léa","Paul","Claire","Louis","Camille"],
 "es": ["Juan","María","Carlos","Ana","Luis","Sofía","Pedro","Laura","Miguel","Lucía","Jorge","Elena"],
 "de": ["Hans","Anna","Peter","Julia","Lukas","Sophie","Felix","Laura","Paul","Lena","Jonas","Marie"],
 "zh": ["王伟","李娜","张敏","刘洋","陈静","杨丽","赵军","黄磊","周杰","吴刚","郑华","孙亮"],
 "ar": ["محمد","أحمد","علي","عمر","خالد","يوسف","حسن","إبراهيم","مصطفى","سعيد","كريم","طارق"],   # male names only (verbs are masculine)
 "hi": ["राम","श्याम","सीता","गीता","मोहन","रोहन","अमित","सुनीता","विजय","अनिल","राधा","किरण"],
}

# n1, n2 = first two mentions; n3 = repeated subject (S). Target = the other name (IO). Final word constant per language.
TEMPLATES = {
 "en": ["Then, {n1} and {n2} went to the shop. {n3} gave a drink to",
        "After lunch, {n1} and {n2} went to the park. {n3} gave a book to",
        "When {n1} and {n2} got to the office, {n3} gave a letter to",
        "Yesterday {n1} and {n2} were at the school. {n3} handed a pen to"],
 "fr": ["Ensuite, {n1} et {n2} sont allés au magasin. {n3} a donné une boisson à",
        "Après le déjeuner, {n1} et {n2} sont allés au parc. {n3} a donné un livre à",
        "Quand {n1} et {n2} sont arrivés au bureau, {n3} a donné une lettre à",
        "Hier, {n1} et {n2} étaient à l'école. {n3} a tendu un stylo à"],
 "es": ["Luego, {n1} y {n2} fueron a la tienda. {n3} le dio una bebida a",
        "Después del almuerzo, {n1} y {n2} fueron al parque. {n3} le dio un libro a",
        "Cuando {n1} y {n2} llegaron a la oficina, {n3} le dio una carta a",
        "Ayer {n1} y {n2} estaban en la escuela. {n3} le pasó un bolígrafo a"],
 "de": ["Dann gingen {n1} und {n2} zum Laden. {n3} gab ein Getränk an",
        "Nach dem Mittagessen gingen {n1} und {n2} in den Park. {n3} gab ein Buch an",
        "Als {n1} und {n2} im Büro ankamen, gab {n3} einen Brief an",
        "Gestern waren {n1} und {n2} in der Schule. {n3} reichte einen Stift an"],
 "zh": ["然后，{n1}和{n2}去了商店。{n3}把一杯饮料给",
        "午饭后，{n1}和{n2}去了公园。{n3}把一本书给",
        "当{n1}和{n2}到了办公室，{n3}把一封信给",
        "昨天{n1}和{n2}在学校。{n3}把一支笔给"],
 "ar": ["ثم ذهب {n1} مع {n2} إلى المتجر. أعطى {n3} مشروبا إلى",
        "بعد الغداء ذهب {n1} مع {n2} إلى الحديقة. أعطى {n3} كتابا إلى",
        "عندما وصل {n1} مع {n2} إلى المكتب، أعطى {n3} رسالة إلى",
        "أمس كان {n1} مع {n2} في المدرسة. أعطى {n3} قلما إلى"],
 "hi": ["फिर {n1} और {n2} दुकान गए। {n3} ने",
        "दोपहर के खाने के बाद {n1} और {n2} पार्क गए। {n3} ने",
        "जब {n1} और {n2} दफ़्तर पहुँचे, तब {n3} ने",
        "कल {n1} और {n2} स्कूल में थे। {n3} ने"],
}

# minimal pairs for the first-token-collision check (number: en/fr/es/de; gender: ar/hi; zh has no agreement)
AGREE_PAIRS = {
 "en": [("eats","eat"),("runs","run"),("writes","write"),("plays","play"),("reads","read")],
 "fr": [("mange","mangent"),("joue","jouent"),("écrit","écrivent"),("lit","lisent"),("parle","parlent")],
 "es": [("come","comen"),("juega","juegan"),("escribe","escriben"),("lee","leen"),("habla","hablan")],
 "de": [("isst","essen"),("spielt","spielen"),("schreibt","schreiben"),("liest","lesen"),("spricht","sprechen")],
 "ar": [("يكتب","تكتب"),("يذهب","تذهب"),("يلعب","تلعب"),("يقرأ","تقرأ"),("يأكل","تأكل")],
 "hi": [("खाता","खाती"),("जाता","जाती"),("पढ़ता","पढ़ती"),("लिखता","लिखती"),("खेलता","खेलती")],
}

# data sanity: final token must be constant within a language (Exp16 lesson)
for lang, ts in TEMPLATES.items():
    rend = [t.format(n1="X", n2="Y", n3="Y") for t in ts]
    lasts = {r[-1] if lang == "zh" else r.split()[-1] for r in rend}
    assert len(lasts) == 1, (lang, lasts)
print("config ok; models:", list(MODELS), "| langs:", LANGS)

config ok; models: ['bloom-1b7', 'bloom-7b1'] | langs: ['en', 'fr', 'es', 'de', 'zh', 'ar', 'hi']


In [2]:
# CELL 2 — helpers (tokenizer-only; no GPU)
def enc(tok, text):
    return tok(text, add_special_tokens=False)["input_ids"]

def new_tokens(tok, prefix, cont):
    """Tokens that `cont` adds after `prefix`. None if the boundary merge changes the prefix tokens."""
    a, b = enc(tok, prefix), enc(tok, prefix + cont)
    return b[len(a):] if b[:len(a)] == a else None

def render(lang, ti, n1, n2, n3):
    return TEMPLATES[lang][ti].format(n1=n1, n2=n2, n3=n3)

def usable_names(tok, lang, pool):
    """Names that are exactly one token at the answer position under EVERY template of `lang`."""
    f1, f2 = pool[0], pool[1]
    sep, ok = SEP.get(lang, " "), []
    for nm in pool:
        good = True
        for ti in range(len(TEMPLATES[lang])):
            nt = new_tokens(tok, render(lang, ti, f1, f2, f2), sep + nm)
            if nt is None or len(nt) != 1:
                good = False; break
        if good: ok.append(nm)
    return ok
print("helpers ok")

helpers ok


In [3]:
# CELL 3 — Checks 1+2: name tokenization and agreement-form collisions
import pandas as pd
from transformers import AutoTokenizer

TOK, USABLE, COLLIDE = {}, {}, {}
for key, path in MODELS.items():
    tok = AutoTokenizer.from_pretrained(path); TOK[key] = tok
    for lang in LANGS:
        USABLE[(key, lang, "native")] = usable_names(tok, lang, NAMES[lang])
        if lang != "en":
            USABLE[(key, lang, "en_names")] = usable_names(tok, lang, NAMES["en"])
    for lang, pairs in AGREE_PAIRS.items():
        ctx = PARALLEL[lang][0].rstrip("।.。")
        hit = 0
        for f1, f2 in pairs:
            a, b = new_tokens(tok, ctx, " " + f1), new_tokens(tok, ctx, " " + f2)
            if a and b and a[0] == b[0]: hit += 1
        COLLIDE[(key, lang)] = hit / len(pairs)

rows = []
for key in MODELS:
    for lang in LANGS:
        rows.append(dict(model=key, lang=lang,
                         native_ok=len(USABLE[(key, lang, "native")]),
                         en_names_ok=(len(USABLE[(key, lang, "en_names")]) if lang != "en" else float("nan")),
                         of=len(NAMES[lang]),
                         agree_first_token_collision=COLLIDE.get((key, lang), float("nan"))))
df_tok = pd.DataFrame(rows)
print(df_tok.to_string(index=False))
print(f"\nGate: names_ok >= {MIN_NAMES}. Collision > 0 means the two verb forms cannot be scored at the first token (Exp15b Arabic failure).")

/home/anwoy/miniconda3/envs/ke/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


    model lang  native_ok  en_names_ok  of  agree_first_token_collision
bloom-1b7   en         12          NaN  12                          0.0
bloom-1b7   fr         11         12.0  12                          0.0
bloom-1b7   es         12         12.0  12                          0.0
bloom-1b7   de         12         12.0  12                          0.4
bloom-1b7   zh          0         12.0  12                          NaN
bloom-1b7   ar         12         12.0  12                          0.0
bloom-1b7   hi         10         12.0  12                          0.6
bloom-7b1   en         12          NaN  12                          0.0
bloom-7b1   fr         11         12.0  12                          0.0
bloom-7b1   es         12         12.0  12                          0.0
bloom-7b1   de         12         12.0  12                          0.4
bloom-7b1   zh          0         12.0  12                          NaN
bloom-7b1   ar         12         12.0  12                      

In [4]:
# CELL 4 — Check 3: ROOTS bytes -> characters / tokens (relative to English)
import statistics as st
def stats(tok, lang):
    texts = PARALLEL[lang]
    bpc = st.mean(len(t.encode("utf-8")) / len(t) for t in texts)
    tpc = st.mean(len(enc(tok, t)) / len(t) for t in texts)
    return bpc, tpc, tpc / bpc            # bytes/char, tokens/char, tokens/byte

rows = []
for key, tok in TOK.items():
    s = {l: stats(tok, l) for l in LANGS}
    for l in LANGS:
        sh = ROOTS_BYTE_SHARE.get(l)
        r = dict(model=key, lang=l, bytes_per_char=round(s[l][0], 2), tokens_per_char=round(s[l][1], 2),
                 tokens_per_byte=round(s[l][2], 3), roots_byte_share=sh)
        if sh is not None and ROOTS_BYTE_SHARE["en"]:
            r["char_share_vs_en"] = round((sh / s[l][0]) / (ROOTS_BYTE_SHARE["en"] / s["en"][0]), 3)
            r["token_share_vs_en"] = round((sh * s[l][2]) / (ROOTS_BYTE_SHARE["en"] * s["en"][2]), 3)
        rows.append(r)
df_share = pd.DataFrame(rows)
print(df_share.to_string(index=False))
print("\nBased on 6 hand-written sentences per language: treat ratios as rough. BLOOM tokenizer rows are the ones to cite; other tokenizers are for comparison only.")

    model lang  bytes_per_char  tokens_per_char  tokens_per_byte  roots_byte_share  char_share_vs_en  token_share_vs_en
bloom-1b7   en            1.00             0.21            0.206             30.03             1.000              1.000
bloom-1b7   fr            1.03             0.21            0.202             12.90             0.417              0.420
bloom-1b7   es            1.02             0.21            0.202             10.85             0.353              0.354
bloom-1b7   de            1.01             0.30            0.296               NaN               NaN                NaN
bloom-1b7   zh            3.00             0.58            0.192             16.16             0.179              0.501
bloom-1b7   ar            1.82             0.26            0.145              4.60             0.084              0.107
bloom-1b7   hi            2.60             0.26            0.098               NaN               NaN                NaN
bloom-7b1   en            1.00          

In [5]:
# CELL 5 — Check 4: IOI baseline (forced choice IO vs S at the final position)
import torch, gc
from transformers import AutoModelForCausalLM
dev = "cuda" if torch.cuda.is_available() else "cpu"

def build_items(lang, pool, n, seed):
    rng, T, items = random.Random(seed), TEMPLATES[lang], []
    for i in range(n):
        io, s = rng.sample(pool, 2)
        order = "ABBA" if (i // len(T)) % 2 == 0 else "BABA"
        n1, n2 = (io, s) if order == "ABBA" else (s, io)
        items.append(dict(prefix=render(lang, i % len(T), n1, n2, s), io=io, s=s, order=order))
    return items

@torch.no_grad()
def margin(model, tok, lang, it, bos):
    ids = enc(tok, it["prefix"])
    if bos is not None: ids = [bos] + ids
    logits = model(torch.tensor([ids], device=dev)).logits[0, -1].float()
    sep = SEP.get(lang, " ")
    io_t = new_tokens(tok, it["prefix"], sep + it["io"])[0]
    s_t = new_tokens(tok, it["prefix"], sep + it["s"])[0]
    return (logits[io_t] - logits[s_t]).item()

BASE = {}
for key, path in MODELS.items():
    try:
        model = AutoModelForCausalLM.from_pretrained(path, torch_dtype=torch.bfloat16).to(dev).eval()
    except Exception as e:
        print("LOAD FAILED", key, e); continue
    tok = TOK[key]; bos = tok.bos_token_id if prepend_bos(key) else None
    for lang in LANGS:
        for cond in (["native"] if lang == "en" else ["native", "en_names"]):
            pool = USABLE[(key, lang, cond)]
            if len(pool) < 2:
                BASE[(key, lang, cond)] = dict(acc=float("nan"), n=0); continue
            items = build_items(lang, pool, N_PROMPTS, SEED)
            m = [margin(model, tok, lang, it, bos) for it in items]
            acc = lambda sel: sum(1 for x, it in zip(m, items) if sel(it) and x > 0) / max(1, sum(1 for it in items if sel(it)))
            BASE[(key, lang, cond)] = dict(acc=acc(lambda it: True), acc_ABBA=acc(lambda it: it["order"] == "ABBA"),
                                           acc_BABA=acc(lambda it: it["order"] == "BABA"),
                                           mean_margin=sum(m) / len(m), n=len(items))
    del model; gc.collect(); torch.cuda.empty_cache()
    print("done", key)

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 293/293 [00:00<00:00, 1950.48it/s]


done bloom-1b7


Loading weights: 100%|██████████| 365/365 [00:01<00:00, 335.53it/s]


done bloom-7b1


In [6]:
# CELL 6 — verdict + save
rows = []
for (key, lang, cond), r in BASE.items():
    n_ok = len(USABLE[(key, lang, cond)])
    acc = r.get("acc", float("nan"))
    gap = abs(r.get("acc_ABBA", float("nan")) - r.get("acc_BABA", float("nan")))
    rows.append(dict(model=key, lang=lang, cond=cond, names_ok=n_ok, acc=round(acc, 3),
                     acc_ABBA=round(r.get("acc_ABBA", float("nan")), 3), acc_BABA=round(r.get("acc_BABA", float("nan")), 3),
                     order_bias=round(gap, 3), mean_margin=round(r.get("mean_margin", float("nan")), 2),
                     PASS=bool(n_ok >= MIN_NAMES and acc >= ACC_GATE)))
df_verdict = pd.DataFrame(rows).sort_values(["model", "lang", "cond"])
print(df_verdict.to_string(index=False))
print("\nNotes: DE on BLOOM is informational (not a ROOTS language). order_bias large => model is copying by position, not tracking IO.")
df_verdict.to_csv(OUT / "verdict.csv", index=False); df_tok.to_csv(OUT / "tokenization.csv", index=False); df_share.to_csv(OUT / "share_conversion.csv", index=False)
json.dump({f"{k[0]}|{k[1]}|{k[2]}": v for k, v in USABLE.items()}, open(OUT / "usable_names.json", "w"), ensure_ascii=False, indent=1)

    model lang     cond  names_ok   acc  acc_ABBA  acc_BABA  order_bias  mean_margin  PASS
bloom-1b7   ar en_names        12 1.000      1.00      1.00        0.00         3.76  True
bloom-1b7   ar   native        12 1.000      1.00      1.00        0.00         3.25  True
bloom-1b7   de en_names        12 0.910      0.95      0.87        0.08         2.73  True
bloom-1b7   de   native        12 0.930      0.96      0.90        0.06         2.42  True
bloom-1b7   en   native        12 1.000      1.00      1.00        0.00         3.35  True
bloom-1b7   es en_names        12 1.000      1.00      1.00        0.00         3.86  True
bloom-1b7   es   native        12 1.000      1.00      1.00        0.00         3.73  True
bloom-1b7   fr en_names        12 1.000      1.00      1.00        0.00         3.79  True
bloom-1b7   fr   native        11 1.000      1.00      1.00        0.00         4.03  True
bloom-1b7   hi en_names        12 0.975      1.00      0.95        0.05         3.23  True